# Create Retrieval Score Plots

Create horizontal barplots comparing retrieval scores between naive baseline and bimodal bridge models

In [ ]:
# Parameters from Snakemake
benchmarks_dir = snakemake.params.benchmarks_dir
model_mappings = snakemake.params.model_mappings
retrieval_metrics = snakemake.params.retrieval_metrics

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from pathlib import Path

In [ ]:
# Load retrieval data for naive baseline and bimodal bridge models
df = {}
for i, model_type in enumerate(model_mappings.keys()):
    retrieval_file = snakemake.input.retrieval_results[i]
    retrieval_data = pd.read_csv(retrieval_file, index_col=0).loc[snakemake.wildcards.test_dataset].to_dict()
    df[model_type] = retrieval_data

df = pd.DataFrame(df)

# Filter to only the metrics we want
df = df.loc[retrieval_metrics]
df

In [ ]:
# Create the plot
fig, ax = plt.subplots(figsize=(10, 8))

# Create horizontal bar plot
x_pos = np.arange(len(df))
bar_width = 0.35
colors = ["#ff7f7f", "#7fff7f"]  # Red for naive baseline, green for bimodal bridge

for i, (model, color) in enumerate(zip(df.columns, colors)):
    values = df[model].values
    bars = ax.barh(
        x_pos + i * bar_width,
        values,
        bar_width,
        label=model.replace('_', ' ').title(),
        color=color,
        alpha=0.8,
    )

    # Add value labels on bars
    for j, (bar, val) in enumerate(zip(bars, values)):
        if not np.isnan(val):
            ax.text(
                bar.get_width() + 0.005,
                bar.get_y() + bar.get_height() / 2,
                f"{val:.3f}",
                ha="left",
                va="center",
                fontsize=10,
            )

# Customize plot
ax.set_yticks(x_pos + bar_width / 2)
ax.set_yticklabels(
    [metric.replace("test/", "").replace("test_retrieval/", "").replace("_", " ").title() for metric in df.index], 
    fontsize=11
)
ax.set_xlabel("Performance Score", fontsize=12)
ax.set_title(
    f'Retrieval Scores Comparison - {snakemake.wildcards.test_dataset.replace("__", " & ").replace("_", " ").title()}',
    fontsize=14,
    fontweight="bold",
)

# Add legend
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=11)

# Grid for better readability
ax.grid(axis="x", alpha=0.3)
ax.set_axisbelow(True)

# Adjust layout and save
plt.tight_layout()
plt.savefig(snakemake.output.plots, dpi=300, bbox_inches="tight")
plt.show()